# Risk-Adjusted Customer Lifetime Value Prediction

**AUTHOR**  
Rodrigo Kang

## Overview

The objective is to estimate a risk-adjusted customer lifetime value (CLV) score from transactional data by integrating customer profitability, behavioural segmentation and customer risk into a unified scoring framework.

The target variable in this notebook is not an independently observed future lifetime value. It is a constructed financial estimate based on calibration-period customer profit, segment-level churn rate and segment-level discount rate. The supervised regression layer is therefore used to learn and operationalise this constructed valuation function, rather than to claim that future realised CLV is directly observed.

This notebook starts from the final output of the churn prediction workflow. RFM segmentation and churn labelling are treated as upstream artefacts rather than recomputed here.

## Libraries and Configuration

The configuration follows the previous customer analytics notebooks. Outputs are saved under a dedicated folder so that tables and figures can be reused directly in the portfolio chapter.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Database connectivity
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Visualization
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Modelling utilities
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Miscellaneous
from pathlib import Path
import warnings

In [2]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

pio.renderers.default = "notebook_connected"

seed = 42
np.random.seed(seed)

outputs_dir = Path("3-customer-value-and-retention-clv-figures")
tables_dir = outputs_dir / "tables"
processed_dir = Path("../data/processed/python")

outputs_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


def clean_filename(name):
    """
    Convert a table or figure name into a simple file-friendly stem.

    Inputs:
    -------
        name : str
            Original table or figure name.

    Outputs:
    --------
        str
            Sanitized filename.

    Author:
    -------
        Rodrigo Kang
    """

    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "-")
        .replace("_", "-")
    )


def save_table(data, filename, index=False):
    """
    Save a dataframe as CSV in the CLV output tables folder.

    Inputs:
    -------
        data : pandas.DataFrame
            Table to export.

        filename : str
            Output filename without extension.

        index : bool, default=False
            Whether to save the dataframe index.

    Outputs:
    --------
        pathlib.Path
            Path to the exported CSV file.

    Author:
    -------
        Rodrigo Kang
    """

    output_path = tables_dir / f"{clean_filename(filename)}.csv"

    index_label = "metric" if index else None
    data.to_csv(
        output_path,
        index=index,
        index_label=index_label
    )

    return output_path


def save_plotly_figure(fig, filename, width=1000, height=650, scale=2):
    """
    Save a Plotly figure as interactive HTML and static PNG.

    Notes:
    ------
        PNG export requires the kaleido package.

    Inputs:
    -------
        fig : plotly.graph_objects.Figure
            Figure to export.

        filename : str
            Output filename without extension.

        width : int, default=1000
            Figure width in pixels.

        height : int, default=650
            Figure height in pixels.

        scale : int, default=2
            Resolution scaling factor for PNG export.

    Outputs:
    --------
        tuple[pathlib.Path, pathlib.Path]
            Paths to the exported HTML and PNG files.

    Author:
    -------
        Rodrigo Kang
    """

    stem = clean_filename(filename)

    html_path = outputs_dir / f"{stem}.html"
    png_path = outputs_dir / f"{stem}.png"

    fig.write_html(html_path)

    try:
        fig.write_image(
            png_path,
            width=width,
            height=height,
            scale=scale
        )
    except Exception as error:
        print(
            "PNG export failed. Install or update kaleido if needed: "
            f"{error}"
        )

    return html_path, png_path


def apply_plotly_layout(fig, title, xaxis_title=None, yaxis_title=None):
    """
    Apply a consistent visual layout to portfolio figures.

    Inputs:
    -------
        fig : plotly.graph_objects.Figure
            Figure to format.

        title : str
            Figure title.

        xaxis_title : str, optional
            Label for the x-axis.

        yaxis_title : str, optional
            Label for the y-axis.

    Outputs:
    --------
        plotly.graph_objects.Figure
            Formatted Plotly figure.

    Author:
    -------
        Rodrigo Kang
    """

    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1000,
        height=650,
        font=dict(
            family="Arial",
            size=14
        ),
        title_font=dict(size=20),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=70, r=40, t=90, b=70)
    )

    if xaxis_title is not None:
        fig.update_xaxes(title_text=xaxis_title)

    if yaxis_title is not None:
        fig.update_yaxes(title_text=yaxis_title)

    return fig

def regression_metrics(y_true, y_pred):
    """
    Calculate regression metrics used throughout the notebook.

    Parameters
    ----------
    y_true : array-like
        Observed target values.
    y_pred : array-like
        Predicted target values.

    Returns
    -------
    dict
        Dictionary with RMSE, MAE and R-squared.

    Author
    ------
    Rodrigo Kang
    """

    mse = mean_squared_error(y_true, y_pred)

    return {
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }

## Upstream Churn Output

The CLV workflow uses the final customer-level output produced by the churn notebook. This keeps the portfolio pipeline modular: segmentation and churn labelling are owned by the previous workflows, while this notebook adds the financial layer.

In [3]:
churn_output_path = processed_dir / "customer-churn-prediction.csv"

if not churn_output_path.exists():
    raise FileNotFoundError(
        "The churn output file was not found. "
        "Run 2-churn-prediction.ipynb before running this notebook."
    )

churn_data = pd.read_csv(churn_output_path)

required_churn_columns = [
    "customer_id",
    "recency",
    "frequency",
    "monetary",
    "average_latency",
    "cluster",
    "segment",
    "rfm_normalized_score",
    "rfm_churn_rule",
    "returned_in_validation",
    "churn",
    "churn_probability",
    "churn_prediction"
]

missing_churn_columns = [
    column for column in required_churn_columns
    if column not in churn_data.columns
]

if missing_churn_columns:
    raise ValueError(
        "The churn output is missing required columns: "
        f"{missing_churn_columns}"
    )

churn_data["customer_id"] = churn_data["customer_id"].astype(int)
churn_data["cluster"] = churn_data["cluster"].astype(str)

churn_data.head()

,customer_id,recency,frequency,monetary,average_latency,cluster,segment,rfm_normalized_score,rfm_churn_rule,returned_in_validation,churn,churn_probability,churn_prediction
0,29906,458,4,87491.3550,91.0,3,Inactive Low-Value Customers,0.775080,1,0,1,0.999668,1
1,29813,458,4,48761.8560,91.0,3,Inactive Low-Value Customers,0.766098,1,0,1,0.999661,1
2,29582,458,4,153310.5787,91.0,3,Inactive Low-Value Customers,0.783043,1,0,1,0.999652,1
3,29749,458,4,137715.8538,91.0,3,Inactive Low-Value Customers,0.781568,1,0,1,0.999652,1
4,29742,458,4,138840.8436,91.0,3,Inactive Low-Value Customers,0.781680,1,0,1,0.999652,1


In [4]:
churn_input_summary = pd.DataFrame({
    "metric": [
        "Customers",
        "Segments",
        "Observed churn rate",
        "Average churn probability"
    ],
    "value": [
        churn_data["customer_id"].nunique(),
        churn_data["segment"].nunique(),
        churn_data["churn"].mean(),
        churn_data["churn_probability"].mean()
    ]
})

save_table(churn_input_summary, "churn-input-summary")

churn_input_summary

,metric,value
0,Customers,6206.000000
1,Segments,9.000000
2,Observed churn rate,0.010474
3,Average churn probability,0.013037


## Database Connection

The AdventureWorks database is used only to retrieve transactional and product-cost information. The customer risk layer is already available from the churn output.

In [5]:
from local_config import DB_USER, DB_PASSWORD

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database="adventureworks",
)

engine = create_engine(connection_url)

In [6]:
query = """
SELECT
    current_database() AS database_name,
    current_user AS user_name,
    current_setting('server_encoding') AS server_encoding,
    current_setting('client_encoding') AS client_encoding;
"""

pd.read_sql(query, engine)

,database_name,user_name,server_encoding,client_encoding
0,adventureworks,postgres,UTF8,UTF8


## Data Extraction

The extraction remains at order-line level. Revenue, cost and product mix are later aggregated to customer level using only transactions available by the end of the calibration window.

In [7]:
query = """
SELECT
    c.customerid AS customer_id,
    soh.salesorderid AS sales_order_id,
    soh.orderdate AS order_date,
    soh.duedate AS due_date,
    soh.shipdate AS ship_date,
    soh.status AS order_status,
    soh.territoryid AS territory_id,
    st.name AS territory_name,
    st.countryregioncode AS country_region_code,
    sod.salesorderdetailid AS sales_order_detail_id,
    sod.productid AS product_id,
    p.name AS product_name,
    pc.name AS product_category,
    ps.name AS product_subcategory,
    sod.orderqty AS order_quantity,
    sod.unitprice AS unit_price,
    sod.unitpricediscount AS unit_price_discount,
    p.standardcost AS standard_cost,
    (
        sod.orderqty
        * sod.unitprice
        * (1 - sod.unitpricediscount)
    ) AS line_revenue,
    (
        sod.orderqty
        * p.standardcost
    ) AS line_cost
FROM sales.customer AS c
INNER JOIN sales.salesorderheader AS soh
    ON c.customerid = soh.customerid
INNER JOIN sales.salesorderdetail AS sod
    ON soh.salesorderid = sod.salesorderid
INNER JOIN production.product AS p
    ON sod.productid = p.productid
LEFT JOIN production.productsubcategory AS ps
    ON p.productsubcategoryid = ps.productsubcategoryid
LEFT JOIN production.productcategory AS pc
    ON ps.productcategoryid = pc.productcategoryid
LEFT JOIN sales.salesterritory AS st
    ON soh.territoryid = st.territoryid;
"""

transactions = pd.read_sql(query, engine)

transactions["order_date"] = pd.to_datetime(transactions["order_date"])

numeric_columns = [
    "order_quantity",
    "unit_price",
    "unit_price_discount",
    "standard_cost",
    "line_revenue",
    "line_cost"
]

transactions[numeric_columns] = transactions[numeric_columns].astype(float)

transactions.head()

,customer_id,sales_order_id,order_date,due_date,ship_date,order_status,territory_id,territory_name,country_region_code,sales_order_detail_id,product_id,product_name,product_category,product_subcategory,order_quantity,unit_price,unit_price_discount,standard_cost,line_revenue,line_cost
0,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,1,776,"Mountain-100 Black, 42",Bikes,Mountain Bikes,1.0,2024.994,0.0,1898.0944,2024.994,1898.0944
1,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,2,777,"Mountain-100 Black, 44",Bikes,Mountain Bikes,3.0,2024.994,0.0,1898.0944,6074.982,5694.2832
2,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,3,778,"Mountain-100 Black, 48",Bikes,Mountain Bikes,1.0,2024.994,0.0,1898.0944,2024.994,1898.0944
3,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,4,771,"Mountain-100 Silver, 38",Bikes,Mountain Bikes,1.0,2039.994,0.0,1912.1544,2039.994,1912.1544
4,29825,43659,2022-05-30,2022-06-11,2022-06-06,5,5,Southeast,US,5,772,"Mountain-100 Silver, 42",Bikes,Mountain Bikes,1.0,2039.994,0.0,1912.1544,2039.994,1912.1544


In [8]:
transaction_summary = pd.DataFrame({
    "metric": [
        "Order lines",
        "Customers",
        "Orders",
        "Products",
        "Minimum order date",
        "Maximum order date",
        "Total revenue",
        "Total cost"
    ],
    "value": [
        transactions.shape[0],
        transactions["customer_id"].nunique(),
        transactions["sales_order_id"].nunique(),
        transactions["product_id"].nunique(),
        transactions["order_date"].min(),
        transactions["order_date"].max(),
        round(transactions["line_revenue"].sum(), 2),
        round(transactions["line_cost"].sum(), 2)
    ]
})

save_table(transaction_summary, "transaction-summary")

transaction_summary

,metric,value
0,Order lines,121317
1,Customers,19119
2,Orders,31465
3,Products,266
4,Minimum order date,2022-05-30 00:00:00
5,Maximum order date,2025-06-29 00:00:00
6,Total revenue,109846381.4
7,Total cost,100474477.77


## Calibration and Validation Windows

The same calibration and validation split used by the churn notebook is reconstructed from the observed transaction dates. Customer features and profitability are computed from calibration transactions only.

In [9]:
min_order_date = transactions["order_date"].min().normalize()
max_order_date = transactions["order_date"].max().normalize()

validation_start = max_order_date - pd.DateOffset(years=1) + pd.Timedelta(days=1)
validation_end = max_order_date

calibration_start = min_order_date
calibration_end = validation_start - pd.Timedelta(days=1)
calibration_reference_date = validation_start

window_summary = pd.DataFrame({
    "window": ["Calibration", "Validation"],
    "start_date": [calibration_start, validation_start],
    "end_date": [calibration_end, validation_end]
})

save_table(window_summary, "window-summary")

window_summary

,window,start_date,end_date
0,Calibration,2022-05-30,2024-06-29
1,Validation,2024-06-30,2025-06-29


In [10]:
calibration_transactions = transactions[
    (transactions["order_date"] >= calibration_start)
    & (transactions["order_date"] <= calibration_end)
].copy()

validation_transactions = transactions[
    (transactions["order_date"] >= validation_start)
    & (transactions["order_date"] <= validation_end)
].copy()

period_summary = pd.DataFrame({
    "period": ["Calibration", "Validation"],
    "order_lines": [
        calibration_transactions.shape[0],
        validation_transactions.shape[0]
    ],
    "customers": [
        calibration_transactions["customer_id"].nunique(),
        validation_transactions["customer_id"].nunique()
    ],
    "orders": [
        calibration_transactions["sales_order_id"].nunique(),
        validation_transactions["sales_order_id"].nunique()
    ],
    "revenue": [
        calibration_transactions["line_revenue"].sum(),
        validation_transactions["line_revenue"].sum()
    ],
    "cost": [
        calibration_transactions["line_cost"].sum(),
        validation_transactions["line_cost"].sum()
    ]
})

save_table(period_summary.round(2), "period-summary")

period_summary.round(2)

,period,order_lines,customers,orders,revenue,cost
0,Calibration,43033,6206,8263,64841795.92,61705272.66
1,Validation,78284,18069,23202,45004585.48,38769205.11


## Exploratory Summary

A short inspection confirms that the economic extraction is complete before customer-level profitability is constructed.

In [11]:
missing_summary = (
    transactions
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_values"})
)

missing_summary["missing_share"] = (
    missing_summary["missing_values"] / len(transactions)
)

save_table(missing_summary, "missing-values")

missing_summary.sort_values("missing_values", ascending=False).head(15)

,column,missing_values,missing_share
0,customer_id,0,0.0
1,sales_order_id,0,0.0
2,order_date,0,0.0
3,due_date,0,0.0
4,ship_date,0,0.0
5,order_status,0,0.0
6,territory_id,0,0.0
7,territory_name,0,0.0
8,country_region_code,0,0.0
9,sales_order_detail_id,0,0.0


In [12]:
revenue_summary = (
    transactions[["line_revenue", "line_cost"]]
    .describe()
    .T
)

revenue_summary["skewness"] = transactions[
    ["line_revenue", "line_cost"]
].skew()

save_table(revenue_summary.round(4), "revenue-cost-summary", index=True)

revenue_summary.round(4)

,count,mean,std,min,25%,50%,75%,max,skewness
line_revenue,121317.0,905.4492,1693.4174,1.3740,24.9900,134.9820,1120.4900,27893.6190,3.9524
line_cost,121317.0,828.1978,1705.5965,0.8565,10.8423,104.7052,1030.9488,38530.3854,4.5896


## Customer Profit Estimation

Customer profit is estimated over the calibration period using product standard cost as a proxy for unit cost. In this setting, customer profit should be interpreted as an accounting approximation of gross contribution margin rather than realised net profit.

This assumption is practical for the AdventureWorks schema, but it should be revisited when historical cost records, fulfilment costs, discounts, returns or service costs are available.

In [13]:
calibration_transactions["line_profit"] = (
    calibration_transactions["line_revenue"]
    - calibration_transactions["line_cost"]
)

customer_profit = (
    calibration_transactions
    .groupby("customer_id")
    .agg(
        customer_profit=("line_profit", "sum"),
        calibration_revenue=("line_revenue", "sum"),
        calibration_cost=("line_cost", "sum"),
        calibration_orders=("sales_order_id", "nunique"),
        calibration_order_lines=("sales_order_detail_id", "count"),
        calibration_quantity=("order_quantity", "sum"),
        calibration_products=("product_id", "nunique"),
        calibration_categories=("product_category", "nunique"),
        first_calibration_purchase=("order_date", "min"),
        last_calibration_purchase=("order_date", "max"),
        territory_name=("territory_name", "first")
    )
    .reset_index()
)

customer_profit["profit_margin"] = np.where(
    customer_profit["calibration_revenue"] > 0,
    customer_profit["customer_profit"] / customer_profit["calibration_revenue"],
    np.nan
)

customer_profit.head()

,customer_id,customer_profit,calibration_revenue,calibration_cost,calibration_orders,calibration_order_lines,calibration_quantity,calibration_products,calibration_categories,first_calibration_purchase,last_calibration_purchase,territory_name,profit_margin
0,11000,2555.9656,5741.96,3185.9944,2,3,3.0,3,2,2022-06-20,2024-06-19,Australia,0.445138
1,11001,2568.8884,5794.92,3226.0316,2,7,7.0,7,3,2022-06-16,2024-06-17,Australia,0.443300
2,11002,2530.8443,5694.98,3164.1357,2,2,2.0,2,1,2022-06-08,2024-06-01,Australia,0.444399
3,11003,2542.2894,5718.95,3176.6606,2,5,5.0,5,3,2022-05-30,2024-06-06,Australia,0.444538
4,11004,2577.8693,5776.95,3199.0807,2,4,4.0,4,2,2022-06-24,2024-06-23,Australia,0.446234


In [14]:
profit_summary = (
    customer_profit[
        [
            "customer_profit",
            "calibration_revenue",
            "calibration_cost",
            "profit_margin"
        ]
    ]
    .describe()
    .T
)

profit_summary["skewness"] = customer_profit[
    [
        "customer_profit",
        "calibration_revenue",
        "calibration_cost",
        "profit_margin"
    ]
].skew()

save_table(profit_summary.round(4), "profit-summary", index=True)

profit_summary.round(4)

,count,mean,std,min,25%,50%,75%,max,skewness
customer_profit,6206.0,505.4017,3879.1653,-55895.3119,296.2834,805.8001,1406.9758,24595.8475,-7.7054
calibration_revenue,6206.0,10448.2430,47361.5644,1.3740,2049.0982,2181.5625,3578.2700,640042.6211,7.8868
calibration_cost,6206.0,9942.8412,49572.8364,0.8565,1251.9813,1554.9479,2171.2942,695937.9330,8.0157
profit_margin,6206.0,0.3332,0.1345,-0.9425,0.2872,0.3822,0.3932,0.6260,-3.0521


Profitability is highly concentrated. The log-scaled visualisation is used to make the body of the distribution visible without removing high-value customers.

In [15]:
fig = px.histogram(
    customer_profit,
    x="customer_profit",
    nbins=80,
    marginal="box"
)

fig.update_traces(
    hovertemplate="Customer profit: %{x:,.2f}<br>Customers: %{y}<extra></extra>"
)

fig = apply_plotly_layout(
    fig,
    title="Distribution of Customer Profit",
    xaxis_title="Customer Profit",
    yaxis_title="Customers"
)

save_plotly_figure(fig, "customer-profit-distribution")
fig.show()

## Customer Segmentation

The RFM cluster and segment labels are inherited from the churn output. This avoids rerunning clustering and keeps the CLV notebook aligned with the upstream segmentation contract.

In [16]:
customer_base = (
    churn_data
    .merge(
        customer_profit,
        on="customer_id",
        how="left"
    )
)

profit_columns = [
    "customer_profit",
    "calibration_revenue",
    "calibration_cost",
    "calibration_orders",
    "calibration_order_lines",
    "calibration_quantity",
    "calibration_products",
    "calibration_categories",
    "profit_margin"
]

for column in profit_columns:
    customer_base[column] = customer_base[column].fillna(0)

customer_base["territory_name"] = customer_base["territory_name"].fillna("Unknown")

customer_base.head()

,customer_id,recency,frequency,monetary,average_latency,cluster,segment,rfm_normalized_score,rfm_churn_rule,returned_in_validation,churn,churn_probability,churn_prediction,customer_profit,calibration_revenue,calibration_cost,calibration_orders,calibration_order_lines,calibration_quantity,calibration_products,calibration_categories,first_calibration_purchase,last_calibration_purchase,territory_name,profit_margin
0,29906,458,4,87491.3550,91.0,3,Inactive Low-Value Customers,0.775080,1,0,1,0.999668,1,-4655.2778,87491.3550,92146.6328,4,69,123.0,30,3,2022-06-30,2023-03-30,Central,-0.053208
1,29813,458,4,48761.8560,91.0,3,Inactive Low-Value Customers,0.766098,1,0,1,0.999661,1,3075.2074,48761.8560,45686.6486,4,19,34.0,10,2,2022-06-30,2023-03-30,Canada,0.063066
2,29582,458,4,153310.5787,91.0,3,Inactive Low-Value Customers,0.783043,1,0,1,0.999652,1,8910.1066,153310.5787,144400.4721,4,64,153.0,23,4,2022-06-30,2023-03-30,Northwest,0.058118
3,29749,458,4,137715.8538,91.0,3,Inactive Low-Value Customers,0.781568,1,0,1,0.999652,1,-10871.3388,137715.8538,148587.1926,4,115,366.0,35,4,2022-06-30,2023-03-30,Southeast,-0.078940
4,29742,458,4,138840.8436,91.0,3,Inactive Low-Value Customers,0.781680,1,0,1,0.999652,1,7625.7784,138840.8436,131215.0652,4,65,161.0,22,4,2022-06-30,2023-03-30,Northwest,0.054925


In [17]:
segment_summary = (
    customer_base
    .groupby(["cluster", "segment"])
    .agg(
        customers=("customer_id", "count"),
        total_profit=("customer_profit", "sum"),
        avg_profit=("customer_profit", "mean"),
        median_profit=("customer_profit", "median"),
        total_revenue=("calibration_revenue", "sum")
    )
    .reset_index()
)

segment_summary["customer_share"] = (
    segment_summary["customers"]
    / segment_summary["customers"].sum()
)

segment_summary = segment_summary.sort_values(
    "total_profit",
    ascending=False
)

save_table(segment_summary.round(4), "segment-profit-summary")

segment_summary.round(4)

,cluster,segment,customers,total_profit,avg_profit,median_profit,total_revenue,customer_share
0,0,Active Minimal Buyers,1767,2.510039e+06,1420.5087,1406.9758,6.261632e+06,0.2847
6,6,Declining Frequent Buyers,1395,1.061137e+06,760.6715,797.1169,3.059576e+06,0.2248
4,4,Frequent Low-Spend Customers,928,6.964590e+05,750.4946,797.1169,2.045041e+06,0.1495
1,1,High Value but Cooling,454,5.715834e+05,1258.9942,1272.0149,2.218323e+06,0.0732
2,2,Inactive Minimal Buyers,686,1.778722e+05,259.2889,287.3577,5.454052e+05,0.1105
8,8,Recent Low-Value Customers,609,1.714430e+05,281.5156,296.2834,5.200473e+05,0.0981
5,5,Champions,41,9.756634e+02,23.7967,21.8411,2.071293e+03,0.0066
7,7,Lost Customers,53,-4.932814e+05,-9307.1967,-6802.9009,2.309353e+06,0.0085
3,3,Inactive Low-Value Customers,273,-1.559704e+06,-5713.2015,-1723.3340,4.788035e+07,0.0440


In [18]:
fig = px.bar(
    segment_summary.sort_values("customers", ascending=True),
    x="customers",
    y="segment",
    orientation="h",
    text="customers"
)

fig.update_traces(
    textposition="outside",
    hovertemplate=(
        "Segment: %{y}<br>"
        "Customers: %{x:,}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Customer Distribution by RFM Segment",
    xaxis_title="Customers",
    yaxis_title="Segment"
)

save_plotly_figure(fig, "customer-distribution-by-segment")
fig.show()

## Segment-Level Churn Estimation

Segment churn rates are estimated from the churn labels already created in the previous notebook. Validation activity is used only through that upstream label, not as a predictor in the CLV model.

Because churn is estimated at segment level from hard churn labels, segments with no observed churn in the validation window receive a churn rate of zero. This is acceptable for a transparent baseline, but the resulting CLV should be read as sensitive to this assumption.

In [19]:
segment_churn_rates = (
    customer_base
    .groupby(["cluster", "segment"])
    .agg(
        customers=("customer_id", "count"),
        churners=("churn", "sum"),
        segment_churn_rate=("churn", "mean"),
        avg_churn_probability=("churn_probability", "mean"),
        returned_rate=("returned_in_validation", "mean")
    )
    .reset_index()
    .sort_values("segment_churn_rate", ascending=False)
)

save_table(segment_churn_rates.round(4), "segment-churn-rates")

segment_churn_rates.round(4)

,cluster,segment,customers,churners,segment_churn_rate,avg_churn_probability,returned_rate
3,3,Inactive Low-Value Customers,273,65,0.2381,0.2828,0.6850
0,0,Active Minimal Buyers,1767,0,0.0000,0.0000,0.8557
1,1,High Value but Cooling,454,0,0.0000,0.0062,0.5088
2,2,Inactive Minimal Buyers,686,0,0.0000,0.0009,0.9461
4,4,Frequent Low-Spend Customers,928,0,0.0000,0.0000,0.8006
5,5,Champions,41,0,0.0000,0.0012,0.4878
6,6,Declining Frequent Buyers,1395,0,0.0000,0.0000,0.9154
7,7,Lost Customers,53,0,0.0000,0.0026,1.0000
8,8,Recent Low-Value Customers,609,0,0.0000,0.0000,0.7947


In [20]:
fig = px.bar(
    segment_churn_rates.sort_values("segment_churn_rate", ascending=True),
    x="segment_churn_rate",
    y="segment",
    orientation="h",
    text=segment_churn_rates.sort_values("segment_churn_rate", ascending=True)["segment_churn_rate"].map("{:.1%}".format)
)

fig.update_traces(
    textposition="outside",
    hovertemplate=(
        "Segment: %{y}<br>"
        "Churn rate: %{x:.2%}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Segment-Level Churn Rate",
    xaxis_title="Churn Rate",
    yaxis_title="Segment"
)

fig.update_xaxes(tickformat=".0%")

save_plotly_figure(fig, "segment-churn-rate")
fig.show()

## Segment-Level Discount Rate Estimation

The segment discount rate is an adaptation inspired by the customer valuation literature. Monthly segment returns are compared against total customer returns during calibration; the resulting beta is transformed into a segment discount rate using a simple baseline-rate plus risk-premium specification.

This is not treated as a strict CAPM implementation. It is a practical risk-adjustment layer for customer valuation. Segments with insufficient monthly observations fall back to a neutral beta, and the minimum discount rate prevents non-positive denominators in the CLV formula.

In [21]:
monthly_segment_profit = (
    calibration_transactions
    .merge(
        customer_base[["customer_id", "cluster", "segment"]],
        on="customer_id",
        how="inner"
    )
    .assign(month=lambda data: data["order_date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "cluster", "segment"])
    .agg(monthly_profit=("line_profit", "sum"))
    .reset_index()
)

monthly_total_profit = (
    monthly_segment_profit
    .groupby("month")
    .agg(total_profit=("monthly_profit", "sum"))
    .reset_index()
)

segment_profit_matrix = (
    monthly_segment_profit
    .pivot_table(
        index="month",
        columns="segment",
        values="monthly_profit",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index()
)

total_profit_series = monthly_total_profit.set_index("month")["total_profit"].sort_index()

segment_returns = segment_profit_matrix.pct_change().replace([np.inf, -np.inf], np.nan)
total_returns = total_profit_series.pct_change().replace([np.inf, -np.inf], np.nan)

segment_returns = segment_returns.clip(
    lower=segment_returns.quantile(0.05),
    upper=segment_returns.quantile(0.95),
    axis=1
)

total_returns = total_returns.clip(
    lower=total_returns.quantile(0.05),
    upper=total_returns.quantile(0.95)
)

monthly_segment_profit.head()

,month,cluster,segment,monthly_profit
0,2022-05-01,0,Active Minimal Buyers,7196.5986
1,2022-05-01,1,High Value but Cooling,3307.4110
2,2022-05-01,2,Inactive Minimal Buyers,212.3916
3,2022-05-01,3,Inactive Low-Value Customers,-1232.0621
4,2022-06-01,0,Active Minimal Buyers,159892.9430


In [22]:
baseline_discount_rate = 0.10
risk_premium = 0.05
minimum_discount_rate = 0.01

market_return_variance = total_returns.dropna().var()

discount_rows = []

for segment in segment_returns.columns:
    aligned_data = pd.concat(
        [
            segment_returns[segment].rename("segment_return"),
            total_returns.rename("total_return")
        ],
        axis=1
    ).dropna()

    if len(aligned_data) < 3 or market_return_variance == 0:
        beta = 1.0
    else:
        beta = (
            aligned_data["segment_return"]
            .cov(aligned_data["total_return"])
            / market_return_variance
        )

    discount_rate = max(
        minimum_discount_rate,
        baseline_discount_rate + beta * risk_premium
    )

    discount_rows.append({
        "segment": segment,
        "beta": beta,
        "segment_discount_rate": discount_rate,
        "observed_months": len(aligned_data)
    })

segment_discount_rates = pd.DataFrame(discount_rows)

segment_discount_rates = (
    customer_base[["cluster", "segment"]]
    .drop_duplicates()
    .merge(segment_discount_rates, on="segment", how="left")
)

segment_discount_rates["beta"] = segment_discount_rates["beta"].fillna(1.0)
segment_discount_rates["segment_discount_rate"] = (
    segment_discount_rates["segment_discount_rate"]
    .fillna(baseline_discount_rate + risk_premium)
)

segment_discount_rates = segment_discount_rates.sort_values(
    "segment_discount_rate",
    ascending=False
)

save_table(segment_discount_rates.round(4), "segment-discount-rates")

segment_discount_rates.round(4)

,cluster,segment,beta,segment_discount_rate,observed_months
6,0,Active Minimal Buyers,1.2740,0.1637,13
3,7,Lost Customers,1.0000,0.1500,1
1,1,High Value but Cooling,0.4828,0.1241,25
2,5,Champions,0.4595,0.1230,7
8,8,Recent Low-Value Customers,0.2760,0.1138,6
0,3,Inactive Low-Value Customers,0.1580,0.1079,25
5,2,Inactive Minimal Buyers,0.0590,0.1029,20
4,4,Frequent Low-Spend Customers,-0.4453,0.0777,5
7,6,Declining Frequent Buyers,-1.8314,0.0100,9


In [23]:
discount_assumptions = pd.DataFrame({
    "assumption": [
        "Baseline discount rate",
        "Risk premium multiplier",
        "Minimum discount rate",
        "Return winsorisation"
    ],
    "value": [
        baseline_discount_rate,
        risk_premium,
        minimum_discount_rate,
        "5th to 95th percentile"
    ]
})

save_table(discount_assumptions, "discount-rate-assumptions")

discount_assumptions

,assumption,value
0,Baseline discount rate,0.1
1,Risk premium multiplier,0.05
2,Minimum discount rate,0.01
3,Return winsorisation,5th to 95th percentile


In [24]:
fig = px.bar(
    segment_discount_rates.sort_values("segment_discount_rate", ascending=True),
    x="segment_discount_rate",
    y="segment",
    orientation="h",
    text=segment_discount_rates.sort_values("segment_discount_rate", ascending=True)["segment_discount_rate"].map("{:.1%}".format)
)

fig.update_traces(
    textposition="outside",
    hovertemplate=(
        "Segment: %{y}<br>"
        "Discount rate: %{x:.2%}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Segment-Level Discount Rate",
    xaxis_title="Discount Rate",
    yaxis_title="Segment"
)

fig.update_xaxes(tickformat=".0%")

save_plotly_figure(fig, "segment-discount-rate")
fig.show()

## Customer Lifetime Value Construction

The CLV target combines individual calibration-period profit with segment-level churn and discount rates. The resulting target is a constructed financial outcome, not an independently observed future value.

Because the profit component is accumulated over the calibration window rather than separately annualised, the resulting CLV is best interpreted as a relative monetary valuation score for ranking and prioritisation, not as a literal audited lifetime-profit measure.

In [25]:
customer_base = (
    customer_base
    .merge(
        segment_churn_rates[
            ["cluster", "segment", "segment_churn_rate"]
        ],
        on=["cluster", "segment"],
        how="left"
    )
    .merge(
        segment_discount_rates[
            ["cluster", "segment", "beta", "segment_discount_rate"]
        ],
        on=["cluster", "segment"],
        how="left"
    )
)

customer_base["clv_denominator"] = (
    customer_base["segment_discount_rate"]
    + customer_base["segment_churn_rate"]
)

if (customer_base["clv_denominator"] <= 0).any():
    raise ValueError("CLV denominator must be positive for every customer.")

customer_base["constructed_clv"] = (
    customer_base["customer_profit"]
    * (1 + customer_base["segment_discount_rate"])
    / customer_base["clv_denominator"]
)

customer_base.head()

,customer_id,recency,frequency,monetary,average_latency,cluster,segment,rfm_normalized_score,rfm_churn_rule,returned_in_validation,churn,churn_probability,churn_prediction,customer_profit,calibration_revenue,calibration_cost,calibration_orders,calibration_order_lines,calibration_quantity,calibration_products,calibration_categories,first_calibration_purchase,last_calibration_purchase,territory_name,profit_margin,segment_churn_rate,beta,segment_discount_rate,clv_denominator,constructed_clv
0,29906,458,4,87491.3550,91.0,3,Inactive Low-Value Customers,0.775080,1,0,1,0.999668,1,-4655.2778,87491.3550,92146.6328,4,69,123.0,30,3,2022-06-30,2023-03-30,Central,-0.053208,0.238095,0.158016,0.107901,0.345996,-14906.489331
1,29813,458,4,48761.8560,91.0,3,Inactive Low-Value Customers,0.766098,1,0,1,0.999661,1,3075.2074,48761.8560,45686.6486,4,19,34.0,10,2,2022-06-30,2023-03-30,Canada,0.063066,0.238095,0.158016,0.107901,0.345996,9847.005543
2,29582,458,4,153310.5787,91.0,3,Inactive Low-Value Customers,0.783043,1,0,1,0.999652,1,8910.1066,153310.5787,144400.4721,4,64,153.0,23,4,2022-06-30,2023-03-30,Northwest,0.058118,0.238095,0.158016,0.107901,0.345996,28530.716034
3,29749,458,4,137715.8538,91.0,3,Inactive Low-Value Customers,0.781568,1,0,1,0.999652,1,-10871.3388,137715.8538,148587.1926,4,115,366.0,35,4,2022-06-30,2023-03-30,Southeast,-0.078940,0.238095,0.158016,0.107901,0.345996,-34810.703635
4,29742,458,4,138840.8436,91.0,3,Inactive Low-Value Customers,0.781680,1,0,1,0.999652,1,7625.7784,138840.8436,131215.0652,4,65,161.0,22,4,2022-06-30,2023-03-30,Northwest,0.054925,0.238095,0.158016,0.107901,0.345996,24418.217181


In [26]:
clv_summary = (
    customer_base[
        [
            "constructed_clv",
            "customer_profit",
            "segment_churn_rate",
            "segment_discount_rate"
        ]
    ]
    .describe()
    .T
)

clv_summary["skewness"] = customer_base[
    [
        "constructed_clv",
        "customer_profit",
        "segment_churn_rate",
        "segment_discount_rate"
    ]
].skew()

save_table(clv_summary.round(4), "constructed-clv-summary", index=True)

clv_summary.round(4)

,count,mean,std,min,25%,50%,75%,max,skewness
constructed_clv,6206.0,21699.3861,34402.6970,-325016.3995,3174.2851,10001.9081,21123.0393,93380.9236,-0.2336
customer_profit,6206.0,505.4017,3879.1653,-55895.3119,296.2834,805.8001,1406.9758,24595.8475,-7.7054
segment_churn_rate,6206.0,0.0105,0.0488,0.0000,0.0000,0.0000,0.0000,0.2381,4.4484
segment_discount_rate,6206.0,0.0989,0.0560,0.0100,0.0777,0.1079,0.1637,0.1637,-0.4526


In [27]:
fig = px.histogram(
    customer_base,
    x="constructed_clv",
    nbins=80,
    marginal="box"
)

fig.update_traces(
    hovertemplate="Constructed CLV: %{x:,.2f}<br>Customers: %{y}<extra></extra>"
)

fig = apply_plotly_layout(
    fig,
    title="Distribution of Constructed CLV",
    xaxis_title="Constructed CLV",
    yaxis_title="Customers"
)

save_plotly_figure(fig, "constructed-clv-distribution")
fig.show()

## Feature Engineering

The predictor matrix uses only variables available at the end of the calibration window. Variables used directly to construct the target, such as customer profit, segment churn rate and segment discount rate, are excluded from the supervised feature set.

Some retained predictors remain economically close to the target construction, such as revenue, cost, margin, average profit per order and segment membership. This is intentional for a scoring model whose purpose is to operationalise a constructed valuation function, but it should not be interpreted as independent evidence of future realised customer value.

In [28]:
customer_base["customer_tenure_days"] = (
    calibration_reference_date
    - pd.to_datetime(customer_base["first_calibration_purchase"])
).dt.days

customer_base["days_since_last_purchase"] = (
    calibration_reference_date
    - pd.to_datetime(customer_base["last_calibration_purchase"])
).dt.days

customer_base["avg_order_value"] = np.where(
    customer_base["calibration_orders"] > 0,
    customer_base["calibration_revenue"] / customer_base["calibration_orders"],
    0
)

customer_base["avg_profit_per_order"] = np.where(
    customer_base["calibration_orders"] > 0,
    customer_base["customer_profit"] / customer_base["calibration_orders"],
    0
)

customer_base["avg_quantity_per_order"] = np.where(
    customer_base["calibration_orders"] > 0,
    customer_base["calibration_quantity"] / customer_base["calibration_orders"],
    0
)

customer_base["revenue_per_product"] = np.where(
    customer_base["calibration_products"] > 0,
    customer_base["calibration_revenue"] / customer_base["calibration_products"],
    0
)

customer_base["orders_per_month"] = np.where(
    customer_base["customer_tenure_days"] > 0,
    customer_base["calibration_orders"] / (customer_base["customer_tenure_days"] / 30.4375),
    0
)

customer_base["quantity_per_month"] = np.where(
    customer_base["customer_tenure_days"] > 0,
    customer_base["calibration_quantity"] / (customer_base["customer_tenure_days"] / 30.4375),
    0
)

customer_base[
    [
        "customer_tenure_days",
        "days_since_last_purchase",
        "avg_order_value",
        "avg_profit_per_order",
        "orders_per_month"
    ]
].head()

,customer_tenure_days,days_since_last_purchase,avg_order_value,avg_profit_per_order,orders_per_month
0,731,458,21872.838750,-1163.81945,0.166553
1,731,458,12190.464000,768.80185,0.166553
2,731,458,38327.644675,2227.52665,0.166553
3,731,458,34428.963450,-2717.83470,0.166553
4,731,458,34710.210900,1906.44460,0.166553


In [29]:
target_column = "constructed_clv"

numeric_features = [
    "recency",
    "frequency",
    "monetary",
    "average_latency",
    "rfm_normalized_score",
    "churn_probability",
    "calibration_revenue",
    "calibration_cost",
    "calibration_orders",
    "calibration_order_lines",
    "calibration_quantity",
    "calibration_products",
    "calibration_categories",
    "profit_margin",
    "customer_tenure_days",
    "days_since_last_purchase",
    "avg_order_value",
    "avg_profit_per_order",
    "avg_quantity_per_order",
    "revenue_per_product",
    "orders_per_month",
    "quantity_per_month"
]

categorical_features = [
    "cluster",
    "segment",
    "territory_name"
]

excluded_columns = [
    "customer_id",
    "customer_profit",
    "segment_churn_rate",
    "segment_discount_rate",
    "beta",
    "clv_denominator",
    "constructed_clv",
    "churn",
    "churn_prediction",
    "rfm_churn_rule",
    "returned_in_validation",
    "first_calibration_purchase",
    "last_calibration_purchase"
]

feature_columns = numeric_features + categorical_features

model_data = customer_base.dropna(subset=[target_column]).copy()

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

X.head()

,recency,frequency,monetary,average_latency,rfm_normalized_score,churn_probability,calibration_revenue,calibration_cost,calibration_orders,calibration_order_lines,calibration_quantity,calibration_products,calibration_categories,profit_margin,customer_tenure_days,days_since_last_purchase,avg_order_value,avg_profit_per_order,avg_quantity_per_order,revenue_per_product,orders_per_month,quantity_per_month,cluster,segment,territory_name
0,458,4,87491.3550,91.0,0.775080,0.999668,87491.3550,92146.6328,4,69,123.0,30,3,-0.053208,731,458,21872.838750,-1163.81945,30.75,2916.378500,0.166553,5.121495,3,Inactive Low-Value Customers,Central
1,458,4,48761.8560,91.0,0.766098,0.999661,48761.8560,45686.6486,4,19,34.0,10,2,0.063066,731,458,12190.464000,768.80185,8.50,4876.185600,0.166553,1.415698,3,Inactive Low-Value Customers,Canada
2,458,4,153310.5787,91.0,0.783043,0.999652,153310.5787,144400.4721,4,64,153.0,23,4,0.058118,731,458,38327.644675,2227.52665,38.25,6665.677335,0.166553,6.370640,3,Inactive Low-Value Customers,Northwest
3,458,4,137715.8538,91.0,0.781568,0.999652,137715.8538,148587.1926,4,115,366.0,35,4,-0.078940,731,458,34428.963450,-2717.83470,91.50,3934.738680,0.166553,15.239569,3,Inactive Low-Value Customers,Southeast
4,458,4,138840.8436,91.0,0.781680,0.999652,138840.8436,131215.0652,4,65,161.0,22,4,0.054925,731,458,34710.210900,1906.44460,40.25,6310.947436,0.166553,6.703745,3,Inactive Low-Value Customers,Northwest


In [30]:
feature_sources = {
    **{feature: "rfm_churn_output" for feature in [
        "recency",
        "frequency",
        "monetary",
        "average_latency",
        "rfm_normalized_score",
        "churn_probability",
        "cluster",
        "segment",
        "territory_name"
    ]},
    **{feature: "calibration_transactions" for feature in [
        "calibration_revenue",
        "calibration_cost",
        "calibration_orders",
        "calibration_order_lines",
        "calibration_quantity",
        "calibration_products",
        "calibration_categories",
        "profit_margin",
        "customer_tenure_days",
        "days_since_last_purchase",
        "avg_order_value",
        "avg_profit_per_order",
        "avg_quantity_per_order",
        "revenue_per_product",
        "orders_per_month",
        "quantity_per_month"
    ]}
}

target_proxy_risk = {
    "calibration_revenue": "medium",
    "calibration_cost": "medium",
    "profit_margin": "medium",
    "avg_profit_per_order": "medium",
    "cluster": "medium",
    "segment": "medium"
}

feature_audit = pd.DataFrame({
    "feature": feature_columns,
    "type": [
        "numeric" if feature in numeric_features else "categorical"
        for feature in feature_columns
    ],
    "source": [
        feature_sources.get(feature, "derived")
        for feature in feature_columns
    ],
    "available_at_scoring_time": True,
    "target_proxy_risk": [
        target_proxy_risk.get(feature, "low")
        for feature in feature_columns
    ],
    "used_as_predictor": True
})

save_table(feature_audit, "feature-audit")

feature_audit


,feature,type,source,available_at_scoring_time,target_proxy_risk,used_as_predictor
0,recency,numeric,rfm_churn_output,True,low,True
1,frequency,numeric,rfm_churn_output,True,low,True
2,monetary,numeric,rfm_churn_output,True,low,True
3,average_latency,numeric,rfm_churn_output,True,low,True
4,rfm_normalized_score,numeric,rfm_churn_output,True,low,True
5,churn_probability,numeric,rfm_churn_output,True,low,True
6,calibration_revenue,numeric,calibration_transactions,True,medium,True
7,calibration_cost,numeric,calibration_transactions,True,medium,True
8,calibration_orders,numeric,calibration_transactions,True,low,True
9,calibration_order_lines,numeric,calibration_transactions,True,low,True


## Train/Test Split

A reproducible random split is used for final assessment. Stratification is based on customer segment so that smaller behavioural groups remain represented in both samples.

In [31]:
stratify_values = model_data["segment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=seed,
    stratify=stratify_values
)

split_summary = pd.DataFrame({
    "sample": ["Train", "Test"],
    "customers": [len(y_train), len(y_test)],
    "target_mean": [y_train.mean(), y_test.mean()],
    "target_median": [y_train.median(), y_test.median()]
})

save_table(split_summary.round(4), "split-summary")

split_summary.round(4)

,sample,customers,target_mean,target_median
0,Train,4344,21593.7373,10001.9081
1,Test,1862,21945.8623,10001.9081


## Model Benchmark

The benchmark compares linear, tree-based, distance-based, margin-based and additive regression models. Scaling is applied only to models that require it.

In [32]:
def build_preprocessor(scale_numeric=False, transform_numeric=True):
    """
    Build the preprocessing step used by regression models.

    Inputs:
    -------
        scale_numeric : bool, default=False
            Whether to standardize transformed numerical variables.

        transform_numeric : bool, default=True
            Whether to apply a power transformation to skewed numerical variables.

    Outputs:
    --------
        sklearn.compose.ColumnTransformer
            Column-wise preprocessing object.

    Author:
    -------
        Rodrigo Kang
    """

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    if transform_numeric:
        numeric_steps.append(
            ("power", PowerTransformer(method="yeo-johnson", standardize=False))
        )

    if scale_numeric:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    numeric_pipeline = Pipeline(steps=numeric_steps)

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features)
        ],
        remainder="drop"
    )


def build_regression_pipeline(model, scale_numeric=False, transform_numeric=True):
    """
    Build a preprocessing and regression pipeline.

    Inputs:
    -------
        model
            Scikit-learn compatible regressor.

        scale_numeric : bool, default=False
            Whether numerical variables should be standardized.

        transform_numeric : bool, default=True
            Whether skewed numerical variables should be transformed.

    Outputs:
    --------
        sklearn.pipeline.Pipeline
            End-to-end modelling pipeline.

    Author:
    -------
        Rodrigo Kang
    """

    return Pipeline(steps=[
        (
            "preprocessing",
            build_preprocessor(
                scale_numeric=scale_numeric,
                transform_numeric=transform_numeric
            )
        ),
        ("model", model)
    ])

In [33]:
try:
    from xgboost import XGBRegressor
    xgboost_available = True
except Exception:
    XGBRegressor = None
    xgboost_available = False

try:
    from bartpy.sklearnmodel import SklearnModel as BARTRegressor
    bart_available = True
except Exception:
    BARTRegressor = None
    bart_available = False

model_status = []
models = {}

models["Linear Regression"] = build_regression_pipeline(
    LinearRegression(),
    scale_numeric=True
)

models["Ridge Regression"] = build_regression_pipeline(
    Ridge(random_state=seed),
    scale_numeric=True
)

models["LASSO Regression"] = build_regression_pipeline(
    Lasso(random_state=seed, max_iter=10000),
    scale_numeric=True
)

models["Regression Tree"] = build_regression_pipeline(
    DecisionTreeRegressor(random_state=seed),
    scale_numeric=False
)

models["Random Forest"] = build_regression_pipeline(
    RandomForestRegressor(
        n_estimators=300,
        random_state=seed,
        n_jobs=-1
    ),
    scale_numeric=False
)

if xgboost_available:
    models["XGBoost"] = build_regression_pipeline(
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=-1
        ),
        scale_numeric=False
    )
    model_status.append({"model": "XGBoost", "status": "Available"})
else:
    model_status.append({"model": "XGBoost", "status": "Skipped - package not available"})

models["K-Nearest Neighbors Regression"] = build_regression_pipeline(
    KNeighborsRegressor(),
    scale_numeric=True
)

models["Support Vector Regression"] = build_regression_pipeline(
    SVR(),
    scale_numeric=True
)

if bart_available:
    models["Bayesian Additive Regression Trees"] = build_regression_pipeline(
        BARTRegressor(),
        scale_numeric=False
    )
    model_status.append({"model": "Bayesian Additive Regression Trees", "status": "Available"})
else:
    model_status.append({
        "model": "Bayesian Additive Regression Trees",
        "status": "Skipped - package not available"
    })

for model_name in models:
    if model_name not in [row["model"] for row in model_status]:
        model_status.append({"model": model_name, "status": "Available"})

model_status = pd.DataFrame(model_status)

save_table(model_status, "model-availability")

model_status

,model,status
0,XGBoost,Available
1,Bayesian Additive Regression Trees,Skipped - package not available
2,Linear Regression,Available
3,Ridge Regression,Available
4,LASSO Regression,Available
5,Regression Tree,Available
6,Random Forest,Available
7,K-Nearest Neighbors Regression,Available
8,Support Vector Regression,Available


In [34]:
benchmark_rows = []
fitted_benchmark_models = {}

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    metrics = regression_metrics(y_test, y_pred)

    benchmark_rows.append({
        "model": model_name,
        **metrics
    })

    fitted_benchmark_models[model_name] = pipeline

benchmark = (
    pd.DataFrame(benchmark_rows)
    .sort_values("rmse")
    .reset_index(drop=True)
)

skipped_models = model_status[
    model_status["status"] != "Available"
].copy()

save_table(benchmark.round(4), "model-benchmark")

benchmark.round(4)

,model,rmse,mae,r2
0,Random Forest,1936.5286,345.9793,0.9966
1,XGBoost,2178.4778,473.2691,0.9957
2,Regression Tree,2607.8713,456.9903,0.9938
3,K-Nearest Neighbors Regression,4066.1658,880.0326,0.9850
4,Ridge Regression,8251.9469,4268.8566,0.9384
5,LASSO Regression,8300.8116,4313.9835,0.9377
6,Linear Regression,8316.9955,4332.1693,0.9374
7,Support Vector Regression,35181.7991,19710.3313,-0.1193


## Hyperparameter Optimization

Model selection is based on cross-validation performance. The test set is kept for the final assessment only.

The tuning step is intentionally lightweight: it is designed to select a robust portfolio-ready model, not to exhaustively optimise every candidate estimator.

In [35]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=seed
)

tuning_spaces = {
    "Ridge Regression": {
        "model__alpha": np.logspace(-3, 3, 20)
    },
    "LASSO Regression": {
        "model__alpha": np.logspace(-4, 1, 20)
    },
    "Regression Tree": {
        "model__max_depth": [3, 5, 8, 12, None],
        "model__min_samples_leaf": [5, 10, 25, 50],
        "model__min_samples_split": [10, 25, 50, 100]
    },
    "Random Forest": {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [5, 8, 12, None],
        "model__min_samples_leaf": [2, 5, 10, 25],
        "model__max_features": ["sqrt", 0.5, 0.8]
    },
    "K-Nearest Neighbors Regression": {
        "model__n_neighbors": [5, 10, 20, 35, 50],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    "Support Vector Regression": {
        "model__C": [0.1, 1, 10, 50],
        "model__epsilon": [0.01, 0.1, 0.5, 1.0],
        "model__gamma": ["scale", "auto"]
    }
}

if xgboost_available:
    tuning_spaces["XGBoost"] = {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [2, 3, 4, 5],
        "model__learning_rate": [0.02, 0.05, 0.10],
        "model__subsample": [0.75, 0.85, 1.0],
        "model__colsample_bytree": [0.75, 0.85, 1.0]
    }

if bart_available:
    bart_params = models["Bayesian Additive Regression Trees"].get_params().keys()
    candidate_bart_space = {
        "model__n_trees": [50, 100, 200],
        "model__n_chains": [1, 2],
        "model__n_samples": [100, 200]
    }

    bart_space = {
        parameter: values
        for parameter, values in candidate_bart_space.items()
        if parameter in bart_params
    }

    if bart_space:
        tuning_spaces["Bayesian Additive Regression Trees"] = bart_space

In [36]:
tuning_rows = []
tuned_models = {}

for model_name, param_distributions in tuning_spaces.items():

    if model_name not in models:
        continue

    search = RandomizedSearchCV(
        estimator=models[model_name],
        param_distributions=param_distributions,
        n_iter=min(12, np.prod([len(v) for v in param_distributions.values()])),
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=seed,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    tuned_models[model_name] = search.best_estimator_

    tuning_rows.append({
        "model": model_name,
        "cv_rmse": -search.best_score_,
        "best_params": search.best_params_
    })

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values("cv_rmse")
    .reset_index(drop=True)
)

save_table(tuning_results, "tuning-results")

tuning_results

,model,cv_rmse,best_params
0,Random Forest,3024.755599,"{'model__n_estimators': 200, 'model__min_sampl..."
1,XGBoost,3346.992450,"{'model__subsample': 0.85, 'model__n_estimator..."
2,K-Nearest Neighbors Regression,4387.396362,"{'model__weights': 'distance', 'model__p': 2, ..."
3,Regression Tree,4859.722602,"{'model__min_samples_split': 25, 'model__min_s..."
4,Ridge Regression,9055.991118,{'model__alpha': 0.3359818286283781}
5,LASSO Regression,9060.819610,{'model__alpha': 2.9763514416313193}
6,Support Vector Regression,33184.031068,"{'model__gamma': 'scale', 'model__epsilon': 1...."


In [37]:
untuned_candidates = {
    model_name: fitted_benchmark_models[model_name]
    for model_name in [
        "Linear Regression",
        "Bayesian Additive Regression Trees"
    ]
    if model_name in fitted_benchmark_models
    and model_name not in tuned_models
}

for model_name, fitted_model in untuned_candidates.items():
    y_cv_proxy = fitted_model.predict(X_test)
    metrics = regression_metrics(y_test, y_cv_proxy)

    tuning_results = pd.concat(
        [
            tuning_results,
            pd.DataFrame([{
                "model": model_name,
                "cv_rmse": np.nan,
                "best_params": "Not tuned"
            }])
        ],
        ignore_index=True
    )

candidate_model_name = tuning_results.dropna(subset=["cv_rmse"]).iloc[0]["model"]

final_model = tuned_models[candidate_model_name]
final_model_name = candidate_model_name

final_model_name

'Random Forest'

## Final Model Assessment

The selected model is refitted during cross-validation search and evaluated once on the holdout test set.

Final model selection is driven by cross-validation RMSE rather than by a single holdout ranking. The holdout metrics are used as the final assessment of the selected model.

In [38]:
y_test_pred = final_model.predict(X_test)

final_metrics = pd.DataFrame([
    {
        "model": final_model_name,
        **regression_metrics(y_test, y_test_pred)
    }
])

save_table(final_metrics.round(4), "final-model-metrics")

final_metrics.round(4)

,model,rmse,mae,r2
0,Random Forest,2028.6151,363.8659,0.9963


In [39]:
assessment_data = pd.DataFrame({
    "actual_clv": y_test,
    "predicted_clv": y_test_pred
})

assessment_data["residual"] = (
    assessment_data["actual_clv"]
    - assessment_data["predicted_clv"]
)

assessment_data.head()

,actual_clv,predicted_clv,residual
1778,2812.464293,2812.464293,1.045919e-11
5256,10001.908133,10001.908133,3.274181e-11
1964,11051.293209,11051.293209,-2.182787e-11
3918,10001.908133,10001.908133,3.637979e-11
531,2275.495302,2275.495302,9.549694e-12


In [40]:
fig = px.scatter(
    assessment_data,
    x="actual_clv",
    y="predicted_clv",
    opacity=0.65
)

axis_min = min(
    assessment_data["actual_clv"].min(),
    assessment_data["predicted_clv"].min()
)

axis_max = max(
    assessment_data["actual_clv"].max(),
    assessment_data["predicted_clv"].max()
)

fig.add_trace(
    go.Scatter(
        x=[axis_min, axis_max],
        y=[axis_min, axis_max],
        mode="lines",
        name="Perfect prediction",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Predicted vs Actual CLV",
    xaxis_title="Actual CLV",
    yaxis_title="Predicted CLV"
)

save_plotly_figure(fig, "predicted-vs-actual-clv")
fig.show()

In [41]:
fig = px.scatter(
    assessment_data,
    x="predicted_clv",
    y="residual",
    opacity=0.65
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig = apply_plotly_layout(
    fig,
    title="Residual Plot",
    xaxis_title="Predicted CLV",
    yaxis_title="Residual"
)

save_plotly_figure(fig, "residual-plot")
fig.show()

In [42]:
fig = px.histogram(
    assessment_data,
    x="residual",
    nbins=80,
    marginal="box"
)

fig.update_traces(
    hovertemplate="Residual: %{x:,.2f}<br>Customers: %{y}<extra></extra>"
)

fig = apply_plotly_layout(
    fig,
    title="Residual Distribution",
    xaxis_title="Residual",
    yaxis_title="Customers"
)

save_plotly_figure(fig, "residual-histogram")
fig.show()

In [43]:
error_summary = (
    assessment_data[["actual_clv", "predicted_clv", "residual"]]
    .describe()
    .T
)

error_summary["mae_component"] = assessment_data[
    ["actual_clv", "predicted_clv", "residual"]
].abs().mean()

save_table(error_summary.round(4), "error-summary", index=True)

error_summary.round(4)

,count,mean,std,min,25%,50%,75%,max,mae_component
actual_clv,1862.0,21945.8623,33263.0727,-178980.2684,3174.2851,10001.9081,21126.211,93380.9236,25258.5745
predicted_clv,1862.0,21920.5922,33066.1690,-166065.5745,3174.2851,10001.9081,21354.289,93380.9236,25193.6581
residual,1862.0,25.2701,2029.0026,-34361.5330,-0.0000,0.0000,0.000,20399.4139,363.8659


## Model Interpretation

Feature importance is reported for models that expose native importance measures, such as coefficients for linear models or impurity-based importance for tree-based models.

For tree-based estimators, impurity-based feature importance provides a useful ranking of predictive variables but should not be interpreted as evidence of causal relationships. It may favour continuous features and distribute importance across correlated predictors.

In [44]:
def get_feature_names(pipeline):
    """
    Extract transformed feature names from a fitted regression pipeline.

    Inputs:
    -------
        pipeline : sklearn.pipeline.Pipeline
            Fitted preprocessing and model pipeline.

    Outputs:
    --------
        list
            Transformed feature names.

    Author:
    -------
        Rodrigo Kang
    """

    preprocessing = pipeline.named_steps["preprocessing"]
    feature_names = preprocessing.get_feature_names_out()

    return [
        name
        .replace("numeric__", "")
        .replace("categorical__", "")
        for name in feature_names
    ]


def extract_regression_importance(pipeline):
    """
    Extract feature importance from a fitted regression pipeline.

    Inputs:
    -------
        pipeline : sklearn.pipeline.Pipeline
            Fitted preprocessing and model pipeline.

    Outputs:
    --------
        pandas.DataFrame
            Ranked feature importance table.

    Author:
    -------
        Rodrigo Kang
    """

    model = pipeline.named_steps["model"]
    feature_names = get_feature_names(pipeline)

    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
        importance_type = "model_feature_importance"
    elif hasattr(model, "coef_"):
        values = np.abs(model.coef_).ravel()
        importance_type = "absolute_coefficient"
    else:
        return pd.DataFrame(columns=["feature", "importance", "importance_type", "importance_share"])

    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": values,
        "importance_type": importance_type
    })

    importance = importance.sort_values("importance", ascending=False)
    importance["importance_share"] = (
        importance["importance"] / importance["importance"].sum()
    )

    return importance.reset_index(drop=True)


feature_importance = extract_regression_importance(final_model)

if feature_importance.empty:
    print("The selected model does not expose feature importance.")
else:
    save_table(feature_importance.round(6), "feature-importance")
    feature_importance.head(20)

In [45]:
if not feature_importance.empty:
    fig = px.bar(
        feature_importance.head(20).sort_values("importance", ascending=True),
        x="importance",
        y="feature",
        orientation="h"
    )

    fig.update_traces(
        hovertemplate=(
            "Feature: %{y}<br>"
            "Importance: %{x:.4f}<extra></extra>"
        )
    )

    fig = apply_plotly_layout(
        fig,
        title="Top Feature Importance",
        xaxis_title="Importance",
        yaxis_title="Feature"
    )

    save_plotly_figure(fig, "feature-importance")
    fig.show()

The interpretation should be read as a model diagnostic rather than a causal explanation. In this workflow, high-importance features mainly identify which calibration-period signals help reproduce the constructed CLV target.

## Customer Scoring

The selected model is applied to the complete customer base. Residuals are available for all customers, but only the holdout residuals should be interpreted as out-of-sample model errors.

For the full scoring table, residuals measure approximation error relative to the constructed CLV formula.

In [46]:
customer_base["predicted_clv"] = final_model.predict(
    customer_base[feature_columns]
)

customer_base["residual"] = (
    customer_base["constructed_clv"]
    - customer_base["predicted_clv"]
)

final_customer_table = (
    customer_base[
        [
            "customer_id",
            "cluster",
            "segment",
            "customer_profit",
            "segment_churn_rate",
            "segment_discount_rate",
            "constructed_clv",
            "predicted_clv",
            "residual"
        ]
    ]
    .rename(columns={
        "customer_id": "CustomerID",
        "cluster": "Cluster",
        "segment": "Segment",
        "customer_profit": "Customer Profit",
        "segment_churn_rate": "Segment Churn Rate",
        "segment_discount_rate": "Segment Discount Rate",
        "constructed_clv": "Constructed CLV",
        "predicted_clv": "Predicted CLV",
        "residual": "Residual"
    })
    .sort_values("Predicted CLV", ascending=False)
    .reset_index(drop=True)
)

save_table(final_customer_table.round(4), "final-customer-clv-table")

final_customer_table.head(20)

,CustomerID,Cluster,Segment,Customer Profit,Segment Churn Rate,Segment Discount Rate,Constructed CLV,Predicted CLV,Residual
0,22857,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
1,17184,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
2,17183,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
3,17214,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
4,17889,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
5,17828,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
6,17815,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
7,17813,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
8,17795,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10
9,17793,6,Declining Frequent Buyers,924.5636,0.0,0.01,93380.9236,93380.9236,3.201421e-10


In [47]:
final_output_path = processed_dir / "customer-clv-prediction.csv"

final_customer_table.to_csv(
    final_output_path,
    index=False
)

final_output_path

WindowsPath('../data/processed/python/customer-clv-prediction.csv')

In [48]:
segment_scoring_summary = (
    final_customer_table
    .groupby(["Cluster", "Segment"])
    .agg(
        customers=("CustomerID", "count"),
        total_constructed_clv=("Constructed CLV", "sum"),
        total_predicted_clv=("Predicted CLV", "sum"),
        avg_customer_profit=("Customer Profit", "mean"),
        avg_constructed_clv=("Constructed CLV", "mean"),
        avg_predicted_clv=("Predicted CLV", "mean"),
        avg_residual=("Residual", "mean"),
        churn_rate=("Segment Churn Rate", "mean"),
        discount_rate=("Segment Discount Rate", "mean")
    )
    .reset_index()
    .sort_values("total_predicted_clv", ascending=False)
)

save_table(segment_scoring_summary.round(4), "segment-scoring-summary")

segment_scoring_summary.round(4)

,Cluster,Segment,customers,total_constructed_clv,total_predicted_clv,avg_customer_profit,avg_constructed_clv,avg_predicted_clv,avg_residual,churn_rate,discount_rate
6,6,Declining Frequent Buyers,1395,1.071748e+08,1.072191e+08,760.6715,76827.8169,76859.6038,-31.7869,0.0000,0.0100
0,0,Active Minimal Buyers,1767,1.784336e+07,1.784343e+07,1420.5087,10098.1106,10098.1522,-0.0415,0.0000,0.1637
4,4,Frequent Low-Spend Customers,928,9.655763e+06,9.662603e+06,750.4946,10404.9174,10412.2872,-7.3698,0.0000,0.0777
1,1,High Value but Cooling,454,5.176010e+06,5.160707e+06,1258.9942,11400.9038,11367.1955,33.7083,0.0000,0.1241
2,2,Inactive Minimal Buyers,686,1.905665e+06,1.904456e+06,259.2889,2777.9376,2776.1750,1.7625,0.0000,0.1029
8,8,Recent Low-Value Customers,609,1.677969e+06,1.673549e+06,281.5156,2755.2861,2748.0273,7.2587,0.0000,0.1138
5,5,Champions,41,8.909566e+03,9.945816e+03,23.7967,217.3065,242.5809,-25.2744,0.0000,0.1230
7,7,Lost Customers,53,-3.781824e+06,-3.765480e+06,-9307.1967,-71355.1747,-71046.7894,-308.3853,0.0000,0.1500
3,3,Inactive Low-Value Customers,273,-4.994269e+06,-5.061219e+06,-5713.2015,-18294.0267,-18539.2654,245.2386,0.2381,0.1079


In [49]:
fig = px.bar(
    segment_scoring_summary.sort_values("total_predicted_clv", ascending=True),
    x="total_predicted_clv",
    y="Segment",
    orientation="h",
    text="total_predicted_clv"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate=(
        "Segment: %{y}<br>"
        "Total predicted CLV: %{x:,.2f}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Total Predicted CLV by Customer Segment",
    xaxis_title="Total Predicted CLV",
    yaxis_title="Segment"
)

save_plotly_figure(fig, "total-predicted-clv-by-segment")
fig.show()

In [50]:
fig = px.box(
    final_customer_table,
    x="Segment",
    y="Predicted CLV"
)

fig.update_traces(
    hovertemplate=(
        "Segment: %{x}<br>"
        "Predicted CLV: %{y:,.2f}<extra></extra>"
    )
)

fig = apply_plotly_layout(
    fig,
    title="Predicted CLV Distribution by Segment",
    xaxis_title="Segment",
    yaxis_title="Predicted CLV"
)

fig.update_xaxes(tickangle=35)

save_plotly_figure(fig, "predicted-clv-by-segment")
fig.show()

## Final Remarks

This notebook implemented the final layer of the Retail Analytics workflow. It started from the churn prediction output, added transaction-level revenue and cost from AdventureWorks, estimated customer profit, constructed segment-level churn and discount rates, and used those components to build a risk-adjusted CLV target.

The main finding is that customer value is highly concentrated across segments. Profitability, churn risk and discount-rate assumptions materially affect the resulting CLV, so the target should be interpreted as a structured financial estimate rather than an observed ground truth.

The modelling workflow benchmarks a diverse set of regression models and selects the final model using cross-validation before evaluating it on a holdout set. The resulting scoring table can be used to rank customers, inspect segment-level value concentration and support retention prioritisation.

The main limitations are the use of product standard cost as a proxy for historical cost, the segment-level treatment of churn and discount rates, and the fact that CLV is constructed from calibration-period economics rather than directly observed future margin. Future improvements should test alternative cost assumptions, smoothing of segment-level churn rates, sensitivity to the discount-rate specification, time-based validation, and external deployment constraints.